In [ ]:
"""
Preprocess and scale CIC_ToN_IoT (target domain). Reuse scaler from CICIDS2017,
and calculate coviarance statistics.
"""

### Imports ###
import json
import pandas as pd
from pathlib import Path


In [ ]:
### Import and concatenate CSVs ###

# Creates a Path object pointing to the target-domain CSV directory.
data_dir = Path("data/raw/target")

# Read each CSV with encoding fallback for files that are not UTF-8.
def read_csv_with_fallback(file_path):
    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(file_path, low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("unknown", b"", 0, 1, f"Unable to decode {file_path}")

# Load all target CSV files into a list of DataFrames.
dfs = [read_csv_with_fallback(f) for f in data_dir.glob("*.csv")]
if not dfs:
    raise FileNotFoundError(f"No CSV files found in {data_dir}")

# Concatenate all DataFrames into one target-domain DataFrame.
df = pd.concat(dfs, ignore_index=True)

# Display a quick shape check and preview rows.
print("Dataset shape:", df.shape)
df.head()

In [ ]:
### Feature-space alignment (align features according to predetermined shared feature space) ###

# Canonical feature-space contract shared by source and target pipelines.
FEATURE_LIST_PATH = Path("data/processed/shared_feature_space.json")

# Handle missing file
if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(
        f"Shared feature list not found at {FEATURE_LIST_PATH}. "
        "Create/populate this artifact before running preprocessing."
    )

# Load canonical ordered features used by source preprocessing/model training.
with open(FEATURE_LIST_PATH, "r", encoding="utf-8") as f:
    shared_features = json.load(f)

# Label column can vary slightly by export; detect robustly.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
feature_df = df.drop(columns=[label_col]).copy() if label_col else df.copy()

def align_feature_space(frame, feature_list, fill_missing=False, fill_value=0.0):
    # Validate target columns against the expected shared feature contract.
    feature_list = list(feature_list)
    incoming = set(frame.columns)
    expected = set(feature_list)

    extra = sorted(incoming - expected)
    missing = sorted(expected - incoming)

    # Strict mode (default): fail if contract features are absent.
    if missing and not fill_missing:
        preview = missing[:10]
        raise ValueError(
            f"Missing required features: {preview} (total={len(missing)})"
        )

    # Tolerant mode can be enabled later if needed for sparse target schemas.
    if missing and fill_missing:
        for col in missing:
            frame[col] = fill_value

    # Drop unexpected columns and enforce canonical order for scaler/model input.
    aligned = frame[feature_list].copy()
    return aligned, extra, missing

aligned_X, dropped_extra, missing_cols = align_feature_space(
    feature_df,
    shared_features,
    fill_missing=False,
    fill_value=0.0,
 )

# Reattach labels only if present; label-space alignment runs in the next step.
df = pd.concat([aligned_X, df[[label_col]].reset_index(drop=True)], axis=1) if label_col else aligned_X
print(f"Loaded shared feature list from {FEATURE_LIST_PATH}")
print(f"Aligned target feature count: {len(shared_features)}")
print(f"Dropped extra columns: {len(dropped_extra)}")
print(f"Missing required columns: {len(missing_cols)}")

### Label-space alignment (align labels according to predetermined shared label space) ###
# todo

### Data sanitization ###
# todo

### Scaling (use scaler of CICIDS2017) ###
# todo

### Label encoding (use encoder of CICIDS2017) ###
# todo

### Calculate and export covariance and mean statistics ###
# todo

### Export processed data ###
# todo
